In [13]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gym
import random
import math
from collections import namedtuple, deque
import matplotlib.pyplot as plt

In [14]:
# 환경 설정
env = gym.make("CartPole-v1")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Transition 튜플 정의
Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward', 'priority'))

Device: cpu


In [15]:
# Prioritized Replay Memory 클래스
class PrioritizedReplayMemory:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.memory = []
        self.priorities = np.zeros((capacity,), dtype=np.float32)
        self.position = 0
    
    def push(self, state, action, next_state, reward):
        max_priority = self.priorities.max() if self.memory else 1.0
        if len(self.memory) < self.capacity:
            self.memory.append(Transition(state, action, next_state, reward.item(), max_priority))
        else:
            self.memory[self.position] = Transition(state, action, next_state, reward.item(), max_priority)
        
        self.priorities[self.position] = max_priority
        self.position = (self.position + 1) % self.capacity
    
    def sample(self, batch_size, beta=0.4):
        if len(self.memory) == self.capacity:
            priorities = self.priorities
        else:
            priorities = self.priorities[:self.position]
        
        probs = priorities ** self.alpha
        probs /= probs.sum()
        
        indices = np.random.choice(len(self.memory), batch_size, p=probs)
        transitions = [self.memory[idx] for idx in indices]
        batch = Transition(*zip(*transitions))
        
        weights = (len(self.memory) * probs[indices]) ** (-beta)
        weights /= weights.max()
        
        return batch, indices, torch.tensor(weights, device=device, dtype=torch.float32)
    
    def update_priorities(self, indices, priorities):
        self.priorities[indices] = priorities + 1e-5
    
    def __len__(self):
        return len(self.memory)

# DQN 네트워크 정의
class DQN(nn.Module):
    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        self.layer1 = nn.Linear(n_observations, 128)
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, n_actions)

    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        return self.layer3(x)

In [16]:
# 하이퍼파라미터 설정
BATCH_SIZE = 128
GAMMA = 0.99
EPS_START = 0.9
EPS_END = 0.05
EPS_DECAY = 1000
TAU = 0.005
LR = 1e-4
alpha = 0.6  # PER 우선순위 샘플링 파라미터
beta = 0.4    # IS 가중치 파라미터

target_update = 10
num_episodes = 600

n_actions = env.action_space.n
state, _ = env.reset()
n_observations = len(state)

policy_net = DQN(n_observations, n_actions).to(device)
target_net = DQN(n_observations, n_actions).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)
memory = PrioritizedReplayMemory(10000, alpha)

steps_done = 0

In [17]:
def select_action(state):
    global steps_done
    sample = random.random()
    eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1
    if sample > eps_threshold:
        with torch.no_grad():
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

In [18]:
def optimize_model():
    if len(memory) < BATCH_SIZE:
        return

    transitions, indices, weights = memory.sample(BATCH_SIZE, beta)
    batch = Transition(*zip(*transitions))

    non_final_mask = torch.tensor([s is not None for s in batch.next_state], device=device, dtype=torch.bool)
    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None], dim=0) if any(non_final_mask) else torch.zeros((0, n_observations), device=device)

    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.tensor(batch.reward, device=device, dtype=torch.float32)

    state_action_values = policy_net(state_batch).gather(1, action_batch)

    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        if any(non_final_mask):
            next_state_actions = policy_net(non_final_next_states).max(1)[1].unsqueeze(1)
            next_state_values[non_final_mask] = target_net(non_final_next_states).gather(1, next_state_actions).squeeze(1)

    expected_state_action_values = reward_batch + (GAMMA * next_state_values)

    td_errors = torch.abs(state_action_values.squeeze() - expected_state_action_values).detach()
    memory.update_priorities(indices, td_errors.cpu().numpy())

    loss = (weights * F.smooth_l1_loss(state_action_values, expected_state_action_values.unsqueeze(1), reduction='none')).mean()

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()

In [19]:
for i_episode in range(num_episodes):
    state, _ = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    for t in range(1000):
        action = select_action(state)
        observation, reward, terminated, truncated, _ = env.step(action.item())

        reward = torch.tensor([reward], device=device)
        terminated = bool(terminated)
        truncated = bool(truncated)


        next_state = None if terminated else torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        memory.push(state, action, next_state, reward)
        state = next_state
        optimize_model()

        if terminated or truncated:
            break

    if i_episode % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())

print('Training Complete')


AttributeError: module 'numpy' has no attribute 'bool8'